## Sequential workflow

#### 1) BMI Calculator

In [1]:
from langgraph.graph import StateGraph, START, END  
from typing import TypedDict

In [10]:
# 1) Define state

class BMIState(TypedDict):
    weight_kg:float
    height_m:float
    bmi_value:float
    category: str

In [ ]:
# Node nothing but python function, Here created Nodes

# --------------Calculate_BMI--------------- 

def Calculate_BMI(state:BMIState)-> BMIState:
    weight = state['weight_kg']
    height = state['height_m']
    bmi = weight/(height**2)
    state['bmi_value'] = round(bmi,2)

    return state


# ---------------lable_bmi---------------

def lable_bmi(state:BMIState)-> BMIState:
    bmi = state['bmi_value']

    if bmi<18.5:
        state['category'] = 'Underweight'
    elif 18.5 <= bmi < 25:
        state['category'] = 'Normal'
    elif 25 <= bmi < 30:
        state['category'] = 'Overweight'
    else:
        state['category'] = 'Obses'
    
    return state

In [12]:
# 2) Define your graph
graph = StateGraph(BMIState)

# a) Add nodes to our graph
graph.add_node('Calculate_BMI',Calculate_BMI)
graph.add_node('lable_bmi',lable_bmi)

# b) Add edges 
graph.add_edge(START,'Calculate_BMI')
graph.add_edge('Calculate_BMI','lable_bmi')
graph.add_edge('lable_bmi',END)

# c) Compile the graph
workflow = graph.compile()


In [13]:
initial_state = {'weight_kg':80,'height_m':1.73}
final_state = workflow.invoke(initial_state)
print(final_state)

{'weight_kg': 80, 'height_m': 1.73, 'bmi_value': 26.73, 'category': 'Overweight'}


#### Sequential Workflow with LLM

(Start ---- (Question)) -----> LLM (State : question and answer) -------> State -------->End 

In [16]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

True

In [17]:
class llmstate(TypedDict):

    question : str
    answer : str

In [23]:
def llm_qa(state:llmstate)-> llmstate:
    
    # Take question
    que = state['question']

    # Generate prompt
    prompt = f'Give detailed ans for this {que}'

    # Define model
    model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

    # store ans in answer
    state['answer'] = model.invoke(prompt).content

    return state



In [24]:
# 1) Define Graph

graph = StateGraph(llmstate)

# a) Add node
graph.add_node('llm_qa',llm_qa)

# b) Add edge
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# C) Compile graph
llm_workflow = graph.compile()

In [25]:
initial_state = {'question':'Tell me about LLM'}

final_state = llm_workflow.invoke(initial_state)

print(final_state)

{'question': 'Tell me about LLM', 'answer': 'Let\'s dive deep into the world of Large Language Models (LLMs). This is a vast and rapidly evolving field, so I\'ll break it down into key aspects to give you a comprehensive understanding.\n\n## What are Large Language Models (LLMs)?\n\nAt their core, LLMs are a type of **artificial intelligence (AI) model designed to understand, generate, and manipulate human language**. The "Large" in their name refers to two primary aspects:\n\n1.  **Size of the Model:** LLMs have an enormous number of parameters (billions, even trillions). These parameters are essentially the "weights" and "biases" that the model learns during training, determining how it processes information and makes predictions. More parameters generally allow for more complex pattern recognition and nuanced understanding.\n2.  **Size of the Training Data:** LLMs are trained on truly massive datasets of text and code. This data can include books, articles, websites, code repositori